# Data Acquitation

## Konfigurasi Global

In [ ]:
IS_USING_FFT_DOMAIN = True


ROOT_PROJECT_DATASET = "/mnt/e/Erge-Bearing_RUL/Datasets"
BEARING_VARIATION_FOLDER = ("bearing_1", )

MAIN_BEARING = BEARING_VARIATION_FOLDER[0] # Pilih bearing utama untuk analisis
SAMPLING_RATE = 2560

## Import

In [ ]:
import os
import glob
import re

import numpy as np
from numpy.fft import rfft, rfftfreq

import pandas as pd

import dask.dataframe as dd
import dask.array as da
from dask.diagnostics.progress import ProgressBar

import torch
import torch.optim as optim
import torch.nn as nn

from matplotlib import pyplot as plt
import matplotlib.pyplot as plt

import plotly.graph_objs as go
import plotly.offline as pyo

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

import seaborn as sns

from tqdm import tqdm  # Import tqdm for progress bar

import gc

## Helper Functions

In [ ]:
def temp_output_bearing_root(bearing_name: str) -> str:
    return f"{ROOT_PROJECT_DATASET}/{bearing_name}"

def temp_output_bearing_file_csv(bearing_name: str) -> str:
    return f"{temp_output_bearing_root(bearing_name)}/{bearing_name}_full.csv"

def temp_output_bearing_file_parquet(bearing_name: str) -> str:
    return f"{temp_output_bearing_root(bearing_name)}/{bearing_name}_full.parquet"

def temp_output_bearing_file_fft_parquet(bearing_name: str) -> str:
    return f"{temp_output_bearing_root(bearing_name)}/{bearing_name}_fft.parquet"

## A. Merged Data as Raw Time Domain Data

### Helper Function / Variable

In [ ]:
total_bearings = len(BEARING_VARIATION_FOLDER)

### 1. Definisikan mapper untuk mengubah Format RION ke Format yang digunakan

In [ ]:
# Format yang akan digunakan
COLUMN_DTYPES : dict[str, str] = {
    "time": 'float32', 
    "X": 'float32',
    "Y": 'float32',
    "Z": 'float32',
}


# Mapping Format yang dihasilkan RION ke format yang akan digunakan
RION_TO_USED_FORMAT : dict[str, str] = {
    # Format Lama
    's': "time",
    'm/s2': "X",
    'm/s2.1': "Y",
    'm/s2.2': "Z",

    # Format 6 Juni
    'Time (s)': "time",
    'AI 1/X (m/s2)': "X",
    'AI 2/Y (m/s2)': "Y",
    'AI 3/Z (m/s2)': "Z"
}

### 2. Membuat temporary CSV file (Multiple CSV -> Single CSV)
Single CSV ini akan menggunakan format yang telah terdefinisikan COLUMN_DTYPES

In [ ]:
for i, bearing_name in enumerate(BEARING_VARIATION_FOLDER):
    output_file : str = temp_output_bearing_file_csv(bearing_name)
    csv_files = glob.glob(os.path.join(temp_output_bearing_root(bearing_name), "*", "*.csv"), recursive=True)
    
    exclude_dirs = {"datasets"}

    # Keep files whose parent folder name is NOT excluded
    csv_files = [
        p for p in csv_files
        if os.path.basename(os.path.dirname(p)) not in exclude_dirs
    ]
    
    csv_files.sort(key=lambda x: int(re.search(r'(\d+)', os.path.basename(x)).group(1)))

    last_time_value = 0
    header_written = False  # <-- Track header

    if os.path.exists(output_file):
        print(f"Output file {output_file} already exists. Removing...")
        os.remove(output_file)

    for file_index, csv_file in enumerate(tqdm(csv_files, desc=f"Processing {i}/{total_bearings}", unit="file")):
        first_chunk = True
        for chunk in pd.read_csv(csv_file, dtype=COLUMN_DTYPES, chunksize=8192):
            chunk.rename(columns=RION_TO_USED_FORMAT, inplace=True)

            # Get the first time value in the chunk
            first_time_value = chunk["time"].iloc[0] if not chunk.empty else 0

            # Substract time with first time value
            chunk["time"] = chunk["time"] - first_time_value \
                + last_time_value # Adjust time based on the last value

            # Update last_time_value for the next chunk
            last_time_value = chunk["time"].iloc[-1] if not chunk.empty else last_time_value

            with open(output_file, 'a') as stream_out:
                chunk.to_csv(stream_out, header=not header_written, index=False)
            header_written = True
            first_chunk = False


    print(f"Saved file: {output_file}")

### 3. Mengubah Single CSV menjadi Parquet

In [ ]:
# Convert the temporary CSV files to Parquet format
print("\n\n2. Converting temporary CSV files to Parquet format...")

for i, bearing_name in enumerate(BEARING_VARIATION_FOLDER):
    full_csv_file = temp_output_bearing_file_csv(bearing_name)
    parquet_file = temp_output_bearing_file_parquet(bearing_name)

    print(f"Reading CSV file: {full_csv_file}...")
    df = dd.read_csv(full_csv_file, dtype=COLUMN_DTYPES)
    
    print(f"Converting {full_csv_file} ({i}/{total_bearings}) to Parquet format...")
    df.to_parquet(parquet_file, engine="pyarrow", write_index=False)

    print(f"Saved Parquet file: {parquet_file}")

### 4. Menghapus Single CSV

In [ ]:
# Delete the temporary CSV files
print("\n\n3. Deleting temporary CSV files...")

for i, bearing in enumerate(BEARING_VARIATION_FOLDER):
    full_csv_file = temp_output_bearing_file_csv(bearing)

    if os.path.exists(full_csv_file):
        os.remove(full_csv_file)
        print(f"Deleted {full_csv_file}")
    else:
        print(f"File {full_csv_file} does not exist, skipping deletion.")

## [B. Construct FFT Domain Data]
Hanya berjalan jika variable IS_USING_FFT_DOMAIN bernilai True

### Local Configuration

In [ ]:
INPUT_RAW_DATA_PARQUET = temp_output_bearing_file_parquet(MAIN_BEARING)
OUTPUT_FFT_DATA_PARQUET = temp_output_bearing_file_fft_parquet(MAIN_BEARING)

### Helper Function

In [ ]:
def compute_fft(partition):
    fft_results = []
    
    # Convert the partition to a NumPy array for efficient slicing
    partition_array = partition.to_numpy()
    
    # Process the data in chunks of 2560 rows
    for i in range(0, len(partition_array), SAMPLING_RATE):
        chunk = partition_array[i:i + SAMPLING_RATE]
        
        # Skip if the chunk is incomplete
        if len(chunk) < SAMPLING_RATE:
            continue
        
        # Extract X, Y, Z columns (assuming they exist in the partition)
        x = np.array(chunk[:, partition.columns.get_loc("X")])
        y = np.array(chunk[:, partition.columns.get_loc("Y")])
        z = np.array(chunk[:, partition.columns.get_loc("Z")])
        
        # Compute FFT and frequency bins
        fft_x = np.abs(rfft(x))
        fft_y = np.abs(rfft(y))
        fft_z = np.abs(rfft(z))
        frequencies = rfftfreq(len(x), d=1 / SAMPLING_RATE)
        
        # Create a dictionary for this chunk with frequency bins as keys
        fft_row = {f"FFT_{int(freq)}Hz_X": amp for freq, amp in zip(frequencies, fft_x)}
        fft_row.update({f"FFT_{int(freq)}Hz_Y": amp for freq, amp in zip(frequencies, fft_y)})
        fft_row.update({f"FFT_{int(freq)}Hz_Z": amp for freq, amp in zip(frequencies, fft_z)})
        
        # Include the time from column time 
        if 'time' in partition.columns:
            time_value = chunk[0, partition.columns.get_loc("time")]
            fft_row['time'] = time_value
        else:
            raise ValueError("Column 'time' not found in the partition.")
        
        fft_results.append(fft_row)
    
    # Convert the list of dictionaries to a DataFrame
    return pd.DataFrame(fft_results)

### 1. Tes Ketersediaan Raw Data
Jika tabel terbentuk, maka data berhasil terbaca

In [ ]:
if not IS_USING_FFT_DOMAIN:
    print(f"Skipping")
else:
    print(f"Reading Raw Data Parquet file: {INPUT_RAW_DATA_PARQUET}")
    input_parquet_dataframe = dd.read_parquet(INPUT_RAW_DATA_PARQUET)
    input_parquet_dataframe.head()


### 2. Pembuatan Raw data ke FFT

In [ ]:
if not IS_USING_FFT_DOMAIN:
    print(f"Skipping")
else:
    N_BINS = (SAMPLING_RATE // 2) + 1

    META = {
        **{f"FFT_{i}Hz_X": "float64" for i in range(N_BINS)},  # Adjust range based on FFT output
        **{f"FFT_{i}Hz_Y": "float64" for i in range(N_BINS)},
        **{f"FFT_{i}Hz_Z": "float64" for i in range(N_BINS)},
        "time": "float64",
    }

    FFT_DF = input_parquet_dataframe.map_partitions(compute_fft, meta=META)
    FFT_DF.tail()

    with ProgressBar():
        print(f"Saving FFT results to Parquet file: {OUTPUT_FFT_DATA_PARQUET}")
        FFT_DF.to_parquet(OUTPUT_FFT_DATA_PARQUET, engine="pyarrow", compression="snappy")

    print(f"FFT results saved to {OUTPUT_FFT_DATA_PARQUET}")

### 3. Tes Ketersediaan FFT Data
Jika tabel terbentuk, maka data berhasil terbaca

In [ ]:
if not IS_USING_FFT_DOMAIN:
    print(f"Skipping")
else:
    # Test reading the saved Parquet file
    print("\n\n5. Reading the saved Parquet file to verify...")
    TEST_FFT_DF = dd.read_parquet(OUTPUT_FFT_DATA_PARQUET)
    TEST_FFT_DF.tail()

### Gabage Collector

In [ ]:
if not IS_USING_FFT_DOMAIN:
    print(f"Skipping")
else:
    print("\n\n6. Cleaning up...")
    try:
        del input_parquet_dataframe
        del FFT_DF
    except Exception as e:
        print(f"Error during cleanup: {e}")
    finally:
        gc.collect()

## Selanjutnya "2_Data_Preprocessing"